# OSS Console — Base Detection Model (Colab)

Train **one shared model** for fish + persons-in-water + SAR targets, export to **ONNX** for the browser.

Runtime → **Change runtime type → GPU (T4)** before running. Full plan: `docs/MODEL-TRAINING.md`.


In [ ]:
# 1. Install
!pip -q install ultralytics onnx onnxruntime

## 2. Get the data
- **SAR / PIW:** download **SeaDronesSee** (CC0) from https://www.macvi.org/dataset and upload/mount it.
- **Fish:** in the app's **AI Lab / Dataset** tab → *Export YOLO Dataset (.zip)* and upload it here.
- ⚠️ Skip **AFO** for commercial weights (non-commercial license).

Put each dataset under `/content/data/`. Below assumes YOLO-format folders with `images/` + `labels/`.


In [ ]:
import os, zipfile, glob
os.makedirs('/content/data', exist_ok=True)
# Unzip the app's fish export (upload OSS_dataset.zip first via the Files panel):
for z in glob.glob('/content/*.zip'):
    with zipfile.ZipFile(z) as f: f.extractall('/content/data/'+os.path.basename(z)[:-4])
print('extracted:', os.listdir('/content/data'))

### Convert SeaDronesSee COCO → YOLO (if needed)
SeaDronesSee ships COCO JSON. Ultralytics can convert it to YOLO txt labels:


In [ ]:
# Point to the SeaDronesSee COCO annotation dir, then convert:
from ultralytics.data.converter import convert_coco
# convert_coco(labels_dir='/content/data/seadronessee/annotations', save_dir='/content/data/seadronessee_yolo', use_segments=False)
print('Edit the path above to your SeaDronesSee annotations, then uncomment to run.')

## 3. Define the merged class list + data.yaml
One unified label space across editions. Keep IDs stable — the app maps them to edition vocab.


In [ ]:
classes = [
  'fish','bait_ball','nervous_water',     # fishing
  'person','person_in_water',             # SAR people
  'life_raft','life_jacket','vessel','debris'  # SAR objects
]
import yaml
data = {
  'path': '/content/data',
  'train': 'train/images',   # <- set to your merged train images dir
  'val':   'val/images',     # <- set to your merged val images dir
  'names': {i:n for i,n in enumerate(classes)}
}
open('/content/oss.yaml','w').write(yaml.safe_dump(data))
print(open('/content/oss.yaml').read())

## 4. Train (YOLO11 @ 1280px, water/glare augmentation)
Small targets → large image size. Transfer-learn from a pretrained checkpoint.


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')   # start small; yolo11s/m for more accuracy
model.train(data='/content/oss.yaml', epochs=120, imgsz=1280, batch=8,
            name='oss-base', patience=30,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, fliplr=0.5, mosaic=1.0, degrees=10)

### (optional) Benchmark YOLOv8 — often better on tiny aerial targets
Compare val mAP and keep whichever wins on YOUR data.


In [ ]:
# m8 = YOLO('yolov8n.pt'); m8.train(data='/content/oss.yaml', epochs=120, imgsz=1280, batch=8, name='oss-v8')

## 5. Export ONNX (browser-ready)


In [ ]:
best = 'runs/detect/oss-base/weights/best.pt'
YOLO(best).export(format='onnx', imgsz=1280, simplify=True)
print('ONNX ->', best.replace('.pt','.onnx'))
# Download it from the Files panel, then send it over to wire into the app's neural seam.

## 6. Wire into the app
Drop `best.onnx` into the app's neural-detection slot (ONNX Runtime Web). Ping the team and we'll do this step.

_Tip: re-run as your labeled fish set grows (active learning) — each app export is bigger and the model sharpens._
